# IMS Anomaly Detection — Isolation Forest

Trains a small `IsolationForest` on synthetic signal features so the
ingestion service can flag bursts that look unusual relative to baseline.

**Features** (5):
- `latency_ms`        — observed latency of the signal-emitting call.
- `error_rate`        — fraction of recent calls returning an error.
- `signal_freq_10s`   — signals seen for this component in the last 10 s.
- `payload_size`      — raw payload size in bytes.
- `hour_of_day`       — diurnal context.

Per the assignment instruction, the synthetic dataset is intentionally small
(~5 k rows). We persist the trained model with `joblib` to `model.pkl` for
the backend to load.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
import joblib
rng = np.random.default_rng(42)
N_NORMAL = 4500
N_ANOMALY = 500

In [ ]:
# Normal traffic — low latency, low errors, modest frequency.
normal = pd.DataFrame({
    'latency_ms':       rng.normal(80, 25, N_NORMAL).clip(5, 500),
    'error_rate':       rng.beta(1.2, 30, N_NORMAL),
    'signal_freq_10s':  rng.poisson(5, N_NORMAL),
    'payload_size':     rng.normal(2048, 500, N_NORMAL).clip(64, 8192),
    'hour_of_day':      rng.integers(0, 24, N_NORMAL),
})
# Anomalies — spikes in latency / error / burstiness.
anomaly = pd.DataFrame({
    'latency_ms':       rng.normal(900, 250, N_ANOMALY).clip(200, 5000),
    'error_rate':       rng.beta(8, 2, N_ANOMALY),
    'signal_freq_10s':  rng.poisson(80, N_ANOMALY),
    'payload_size':     rng.normal(2048, 500, N_ANOMALY).clip(64, 8192),
    'hour_of_day':      rng.integers(0, 24, N_ANOMALY),
})
df = pd.concat([normal, anomaly], ignore_index=True).sample(frac=1, random_state=1).reset_index(drop=True)
df.head()

In [ ]:
FEATURES = ['latency_ms','error_rate','signal_freq_10s','payload_size','hour_of_day']
X = df[FEATURES].values
model = IsolationForest(n_estimators=100, contamination=0.1, random_state=42, n_jobs=-1)
model.fit(X)
df['anomaly_score'] = model.decision_function(X)
df['is_anomalous']  = df['anomaly_score'] < -0.05
df['is_anomalous'].mean()

In [ ]:
joblib.dump({'model': model, 'features': FEATURES, 'threshold': -0.05}, 'model.pkl')
print('Saved model.pkl')